In [ ]:
import pandas as pd
import json
import pycountry
import pycountry_convert as pc
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
import h5py  # Make sure h5py initialises properly before pandas ruins it
from IPython.display import display, Markdown, Latex

from emu_renewal.constants import DATA_PATH
from emu_renewal.inputs import get_world_shp
from emu_renewal.outputs import load_targets
from emu_renewal.plotting import plot_multianalysis_fit, add_cont_to_world_geodf, plot_continent_grouping
from emu_renewal.utils import get_countries_by_continent, get_analysis_paths, get_job_commits_df_new

set_matplotlib_formats("svg")

# Purpose
This document presents the calibration to data
for each country, analysis and target indicator for fitting.

In [ ]:
display(Latex(r"\newpage"))

# Country grouping
For this document and the following supplemental files,
we grouped countries according to the following continent groupings.

In [ ]:
#| fig-align: center
world = get_world_shp()
add_cont_to_world_geodf(world)
plt.style.use("default")
plot_continent_grouping(world)

In [ ]:
display(Latex(r"\newpage"))

# Results by continent and country

In [ ]:
#| fig-align: center
plt.style.use("ggplot")
all_countries = json.load(open(DATA_PATH / "config/included.json", "r"))
analysis_paths = get_analysis_paths(["50038825", "50079832", "50083214", "50083223"], all_countries)
countries_by_cont = get_countries_by_continent(all_countries)
for cont, countries in countries_by_cont.items():
    display(Latex(r"\newpage"))
    display(Markdown(f"## {pc.convert_continent_code_to_continent_name(cont)}"))
    for c, country in enumerate(countries):
        country_name = pycountry.countries.lookup(country).name
        if c:
            display(Latex(r"\newpage"))
        display(Markdown(f"### {country_name}"))
        analyses = analysis_paths[country]
        
        no_mob_path = analyses["no_mob"]
    
        spaghs = {a: pd.read_hdf(p / "spaghetti.h5") for a, p in analyses.items()}
        targs = load_targets(no_mob_path)
        display(plot_multianalysis_fit(country, targs, spaghs))

{{< pagebreak >}}

# Commits used for analyses
For reproducibility, the following table gives the (short) commit SHA for each analysis.

In [ ]:
Markdown(get_job_commits_df_new(analysis_paths).to_markdown())